In [0]:
import os, sys
import logging
import datetime
from datetime import datetime, timedelta
from py4j.java_gateway import JavaObject  
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    expr,
    from_utc_timestamp,
    lit,
    max,
    sha2,
    substring,
    to_date,
    when,
    broadcast,
    trim,
)

In [0]:
from functools import wraps
import operator

def reconcile_table(table_name, env_prefix, run_date):
    current_env = spark.sql(f"SELECT COUNT(*) AS countRows FROM `{env_prefix}-cdh`.landed.{table_name} WHERE BusinessDate = '{run_date}'")
    prod = spark.sql(f"SELECT COUNT(*) AS countRows FROM `p-cdh`.landed.{table_name} WHERE BusinessDate = '{run_date}'")
    if spark.catalog.tableExists(f"`{env_prefix}-cdh`.landed.{table_name}"):
        count_current_env, count_prod = current_env.collect()[0]["countRows"], prod.collect()[0]["countRows"]
        if operator.ne(count_current_env, count_prod):
            print(f"Table: {table_name} | Copying Run Date data: {run_date} | prod -> {env_prefix}")
            spark.sql(f"""DELETE FROM `{env_prefix}-cdh`.landed.{table_name} WHERE BusinessDate = '{run_date}'""")
            spark.sql(f"""INSERT INTO `{env_prefix}-cdh`.landed.{table_name} SELECT * FROM `p-cdh`.landed.{table_name} WHERE BusinessDate = '{run_date}'""")
        else:
            print(f"Table: {table_name} | No changes | prod -> {env_prefix}")

    else:
        print(f"Table: {table_name} | Copying Prodf table | prod -> {env_prefix}")
        spark.sql(f"""CREATE OR REPLACE TABLE `{env_prefix}-cdh`.landed.{table_name} CLONE `p-cdh`.landed.{table_name}""")


def create_decorator(function):
    @wraps(function)
    def wrapper(spark, catalog_landed, table_name, table_filters, table_selected_fields, *args, **kwargs):
        nb = sys.modules["__main__"]
        env_prefix  = getattr(nb, "env_prefix", None)
        run_date    = getattr(nb, "run_date", None)

        reconcile_table(table_name, env_prefix, run_date)
        result = function(spark, catalog_landed, table_name, table_filters, table_selected_fields, *args, **kwargs)
        return result
    return wrapper


from pyspark.sql import SparkSession, DataFrame
from typing import Optional
from pyspark.sql.functions import trim

def load_landed_delta_table(
    spark: SparkSession,
    catalog_landed: str,
    table_name: str,
    table_filters: str,
    table_selected_fields: list[str],
) -> DataFrame:
    
    df = (
        spark.read.table(f"{catalog_landed}.{table_name}")
        .filter(table_filters)
        .select(table_selected_fields)
    )

    # Trim all string columns
    for col_name, col_type in df.dtypes:
        if col_type == "string":
            df = df.withColumn(col_name, trim(df[col_name]))

    return df

def load_landed_portfolio_table(
    spark: SparkSession,
    catalog_landed: str,
    table_name: str,
    table_filters: str,
    table_selected_fields: list[str],
) -> DataFrame:
    
    df = (
        spark.read.table(f"{catalog_landed}.{table_name}")
        .filter(table_filters)
        .select(table_selected_fields)
    )

    # Trim all string columns
    for col_name, col_type in df.dtypes:
        if col_type == "string":
            df = df.withColumn(col_name, trim(df[col_name]))

    return df

def create_delivery_delta_table(
    spark: SparkSession,
    delivery_table_path: str,
    cluster_columns: Optional[list[str]],
    schema: str,
) -> None:


    sql_query = f"""
        CREATE TABLE IF NOT EXISTS {delivery_table_path} ( 
        {schema}
        )
        USING DELTA
    """
    
    if cluster_columns:
        sql_query = sql_query + f"CLUSTER BY ({', '.join(cluster_columns)})"

    sql_query = (
        sql_query
        + """
        TBLPROPERTIES (
            'delta.logRetentionDuration' = 'interval 5 weeks',
            'delta.deletedFileRetentionDuration' = 'interval 5 weeks',
            'delta.minReaderVersion' = '3',
            'delta.minWriterVersion' = '7',
            'delta.columnMapping.mode' = 'name',
            'delta.dataSkippingStatsColumns' = 'BusinessDate'
        )
    """
    )

    spark.sql(sql_query)
    spark.sql(
            f"""REPAIR TABLE {delivery_table_path} SYNC METADATA"""   
        )
    
def fetch_max_business_date(spark: SparkSession, catalog_name: str, table_name: str):
    max_business_date = spark.sql(
        f"""SELECT MAX(BusinessDate) AS maxBusinessDate FROM {catalog_name}.{table_name}"""
        ).collect()[0]['maxBusinessDate']
    return max_business_date

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('delivery_zone', 'None', '03-Delivery_zone')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
delivery_zone = dbutils.widgets.get('delivery_zone')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

In [0]:
spark = SparkSession.builder.getOrCreate()
t1 = fetch_max_business_date(spark=spark, catalog_name=catalog_landed, table_name='sme_businessplandata_lbs_pp')
t2 = fetch_max_business_date(spark=spark, catalog_name=curated_zone, table_name='portfolio')
run_date = min(t1, t2)
run_date = datetime.strftime(run_date, '%Y-%m-%d')
print(f'Run Date: {run_date} |Landed catalog: {catalog_landed} |Curated zone: {curated_zone} |Environment: {env_prefix}')

In [0]:
schema = (
    "DebtorCDI String, AgreementNumber String, BusinessPlan_Status String,"
    "BusinessPlanStrategy_Code Int, BusinessPlan_SolutionType String, BusinessPlan_Comments String,"
    "BusinessPlanStrategy_Status String, Confidence String, ProposedBy String, BusinessPlanStrategy_StrategyType String,"
    "BusinessPlanStrategy_Comments String, LastModifiedDate Date, BusinessPlanStrategy_Solution_Type String,"
    "InstallmentTotalAmount  Decimal(19, 2), InstallmentDate Date, PortfolioName String, BusinessDate Date"
    )

cluster_columns = ["BusinessDate"]
date_filter = f"businessdate = {run_date!r}"
delivery_table_path = f"{delivery_zone}.BusinessPlanData_LBS_PP"

# ================== Create Delta Table ==================

logging.info("Creating Delta Table...")
if not spark.catalog.tableExists(delivery_table_path):
    create_delivery_delta_table(
        spark = spark,
        cluster_columns = cluster_columns,
        delivery_table_path = delivery_table_path,
        schema = schema
        ) 
else:
    logging.info("Table already exists")

# ============================== sme_businessplandata_lbs_pp ====================================

business_plan_data_lbs_pp_extra_filters = None
if business_plan_data_lbs_pp_extra_filters is not None:
    business_plan_data_lbs_pp_extra_filters.append(date_filter)
    business_plan_data_lbs_pp_final_filters = " AND ".join(
        business_plan_data_lbs_pp_extra_filters
    )
else:
    business_plan_data_lbs_pp_final_filters = date_filter

business_plan_data_lbs_pp_selected_filters = [
    "DebtorCDI",
    "AgreementNumber",
    "BusinessPlan_Status",
    "BusinessPlanStrategy_Code",
    "BusinessPlan_SolutionType",
    "BusinessPlan_Comments",
    "BusinessPlanStrategy_Status",
    "Confidence",
    "ProposedBy",
    "BusinessPlanStrategy_StrategyType",
    "BusinessPlanStrategy_Comments",
    "LastModifiedDate",
    "BusinessPlanStrategy_Solution_Type",
    "InstallmentTotalAmount",
    "InstallmentDate",
    "PortfolioID",
    "BusinessDate",
    ]


business_plan_data_lbs_pp = load_landed_delta_table(
    spark,
    catalog_landed,  
    "sme_businessplandata_lbs_pp",
    business_plan_data_lbs_pp_final_filters,
    business_plan_data_lbs_pp_selected_filters,
    )

# ================== portfolio ==================

portfolio_extra_filters = None

if portfolio_extra_filters is not None:
    portfolio_extra_filters.append(date_filter)
    portfolio_final_filters = " AND ".join(portfolio_extra_filters)
else:
    portfolio_final_filters = date_filter 

portfolio_selected_fields = [
            "PortfolioName",
            "portfolio_id_sme",
        ]

portfolio = load_landed_portfolio_table( 
            spark,
            curated_zone,  
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

# ================== Joins ==================

business_plan_data_lbs_pp_join_condition = (
    business_plan_data_lbs_pp["PortfolioID"] == portfolio["portfolio_id_sme"]
    )

business_plan_data_lbs_pp = business_plan_data_lbs_pp.join(
    broadcast(portfolio), business_plan_data_lbs_pp_join_condition)

# ================== Target tables schema ==================

target_business_plan_data_lbs_pp = business_plan_data_lbs_pp.select(
    business_plan_data_lbs_pp.DebtorCDI.cast("String"),
    business_plan_data_lbs_pp.AgreementNumber.cast("String"),
    business_plan_data_lbs_pp.BusinessPlan_Status.cast("String"),
    business_plan_data_lbs_pp.BusinessPlanStrategy_Code.cast("Int"),
    business_plan_data_lbs_pp.BusinessPlan_SolutionType.cast("String"), 
    business_plan_data_lbs_pp.BusinessPlan_Comments.cast("String"),
    business_plan_data_lbs_pp.BusinessPlanStrategy_Status.cast("String"), 
    business_plan_data_lbs_pp.Confidence.cast("String"), 
    business_plan_data_lbs_pp.ProposedBy.cast("String"), 
    business_plan_data_lbs_pp.BusinessPlanStrategy_StrategyType.cast("String"), 
    business_plan_data_lbs_pp.BusinessPlanStrategy_Comments.cast("String"), 
    business_plan_data_lbs_pp.LastModifiedDate.cast("Date"), 
    business_plan_data_lbs_pp.BusinessPlanStrategy_Solution_Type.cast("String"), 
    business_plan_data_lbs_pp.InstallmentTotalAmount.cast("Decimal(19, 2)"), 
    business_plan_data_lbs_pp.InstallmentDate.cast("Date"), 
    business_plan_data_lbs_pp.PortfolioName.cast("String"), 
    business_plan_data_lbs_pp.BusinessDate.cast("Date")
)

# ================== Save results ==================

(
target_business_plan_data_lbs_pp.write
.format("delta")
.clusterBy("BusinessDate")
.mode("append")
.saveAsTable(delivery_table_path)
)
print(delivery_table_path)